# Chapter 4: Training Models

### Intro 

- Knowing the under the hood information of a model can help change choose the appropriate <br>
models, right training algorithms, set of hyperparameters, and debug issues/ perform analysis quicker 
- Essential info for understanding, building, and training neural networks 

### Linear Regression 
- equation: 
    $\hat{y} = \theta_0 X_0+ \theta_1 X_1 + \theta_n X_n$
    + $\hat{y}$ = predicted value 
    + $\theta_n$ = parameters or the features of a dataset we are ultimately figuring out <br>
    + $X_n$ = the actual features of a training instance 

- Containerized form 
    $\hat{y}$ = $\ h_0(x)$ = $\vec\theta  * \vec X$
    + dot product 
    + $\ h_0(x) $ = hypothesis function 
    + 0 is the vector of parameters 
    + $ \vec X$ is the feature vector 

- Model performance measurements 
    + MSE = MSE(X, $\ h_0 $) = $\frac{1}{m}$ $\sum_{i=1}^{m}(\theta^T x^i - y^i)^2$
        + m is the amount of training instances 
        + $\theta^T$ is the parameters transposed 
        + $x^i$ is the actual training instance values of those parameters 
        + $y^i$ is the true values for the training instance or known as the labels
    + MSE finds the difference between the predicted and the actual value and squares the difference. The summation then gets averaged. 


### Normal Equation 

- A method to find the parameters that minimize MSE is with the Normal Equation 
    - $\hat{\theta}$ = $ (X^TX)^{-1}$ $X^T y$
    - This is a closed equation, which means the values are computed straight away instead of iterations 
    - Pretty direct by providing the training values and the labels and the answer are the best parameters 
    > from sklearn.linear_models import LinearRegression() <br>
    > lin_reg = LinearRegression() <br>
    > lin_reg.fit(training, labels) <br>
    > line_reg.intercept_, line_reg.coef_

- Computational Complexity 
    + The normal equation computes $ X^TX $, which is (n + 1) * (n + 1) and n is the amount of features 
    + The reasoning for +1 is to account for the bias/y-intercept term. This is a new column not a row 
    + When inverting this result the complexity is about $\ O(n^{2.4})$ to $O(n^3)$
        + If the features get doubled, you multiply the computation time by roughly $\ 2^{2.4} = 5.3$ to $\ 2^{3} = 8$

- Considerations 
    + The normal equation gets very slow when the number of features increase.
    + On the positive side, the growth is linear in terms of size of training set instances, meaning when there are double training instances it takes double the time


### Gradient Descent 
- A optimization algorithm that tweaks parameters repeatedly to minimize cost function or find a minimum 

- In practice 
    + fill the $\theta$ with random values(random initialization)
    + take steps and each step should aim to minimize the cost function until the algorithm converges to a minimum 

- Learning Step
    + The size proportional to the slope of the cost function 
    + The learning step is the size of the steps and it is controlled by the hyperparameter learning rate 
        + if the learning rate is small than many iterations are done to converge, which takes a long time 
        + if the learning rate is too large than minimum could be jumped over and make the algorithm diverge 
- Possible thoughts 
    + What if there are multiple minimums? Won't the minimum not really be the global minimum? 
        + MSE is convex, which means there is at most one global minimum so that is not a possibility 

- Habits 
    + When using gradient descent it is best to scale(Standard Scaler) the features so the global minimum converges quicker 

### Batch Gradient Descent 
- Batch gradient descent uses partial derivatives in terms of the $\theta_i$ of each feature and the calculations is over the **whole training set**
- Gradient descent step: 
    + $\theta^{\text{next step}} = \theta - \eta \nabla_{\theta} \text{MSE}(\theta)$

- quick implementation below 


In [ ]:
import numpy as np
eta = 0.1 # learning rate 
n_epochs = 1000
m = len(X_b) #number of training instances and X_b is the input feature matrix with bias term
np.random.seed(42)
theta = np.random.randn(2, 1) #randomly initialized model parameters 
for epoch in range(n_epochs):
    gradients = 2 / m * X_b.T @ (X_b @ theta - y) # y are the actual target values
    theta = theta - eta * gradients 

- Each iteration over the training set is the called an epoch 
- The gradients variable is the partial in respect to the theta or the features 
- theta changes from what is minus the learning step 
- @ is the matrix multiplication operator 

### Setting the Learning Rate and Epoch Number 
- Use GridSearchCV and its best to reduce the number of epochs so that the grid search can eliminate models that take to long to converge 
- For epochs, set the number high and can stop when the gradient vector becomes tiny, because this signifies the minimum is very close

### Stochastic Gradient Descent(SGD)
- A common issue is that batch gradient descent uses the whole  training set to compute the gradients at each step, which is slow when the training set is large 

- Stochastic Gradient Descent instead randomly picks a single training instance and computes the gradient 
    + This makes it more efficient because there is less data manipulation when computing the gradient 
    + handles very large datasets since it only looks at one instance in memory 
        + This makes it eligible for out core algorithm 
            + out of core is used when the datasets are too large to fit in main memory(RAM) by processing data in mini batches or chunks from disk instead of using all the data at once 
    + Since this algorithm is stochastic or random the cost function will jump and down, decreasing on average 
    + When the minimum is met there will still be bouncing and will never settle down until it's stopped 
    + Once, the algorithm stops the parameters will be good but not optimal 
    + However, if the cost function is irregular SGD will perform better than batch gradient descent 
    + A solution for SGD with a cost function like MSE is to reduce the learning rate as the minimum is converging 

In [ ]:
import numpy as np

n_epochs = 50 
t0, t1 = 5, 50  # learning schedule hyperparameters

def learning_schedule(t):
    return t0 / (t + t1)

np.random.seed(42)
theta = np.random.randn(2,1) #random initialized 

for epochs in range(n_epochs):
    for iteration in range(m): #m is the number of training instances (this does not necessarily mean every instance will be used)
        random_index = np.random.randint(m)
        xi = X_b[random_index:random_index+1]
        yi = y[random_index:random_index+1]
        gradients = 2 * xi.T @ (xi @ theta - yi)
        eta = learning_schedule(epochs * m + iteration)
        theta = theta - eta * gradients


- Every instance is not guaranteed to be used and an instance can be used more than once 
- Recommended to shuffle the training instances to prevent a possible underlying pattern from data order 

### Sklearn implementation 

In [ ]:
from sklearn.linear_model import SGDRegressor 

sgd_reg = SGDRegressor(max_iter=1000, tol=1e-3, penalty=None, eta0=0.1, n_iter_no_change=100, random_state=42)
sgd_reg.fit(X, y.ravel())  # y.ravel() because fit() expects a 1D target 

### The model does not use regularization because penalty=None


### Polynomial Regression 
- Handles data for more complex functions that are more than a straight line 
- Transforming training data to use multi-degree polynomials, Polynomial features is used 


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly_features.fit_transform(X)

### Deciding the Degree 
- How is the degree chosen to prevent underfitting or overfitting?
    + One way would be using cross validation with various degree options 
    + Another method is to look at the visualization of learning curves 
        + Sklearn has the function learning_curve()

In [ ]:
from sklearn.preprocessing import learning_curve
from sklearn.linear_model import LinearRegression
import numpy as np

train_sizes, train_scores, valid_scores = learning_curve(LinearRegression(), X, y,train_sizes=np.linspace(0.01, 1.0, 40), cv=5, scoring='neg_mean_squared_error')
train_errors = -train_scores.mean(axis=1)
valid_errors = -valid_scores.mean(axis=1)